In [0]:
import json
import os
from datetime import datetime
import random
from openai import OpenAI

# =============================================================================
# OpenAI Connection Setup via Databricks Secrets
# =============================================================================

try:
    # Adjust these string values to match your specific secret scope and keys
    SECRET_SCOPE = "openai"
    API_KEY_NAME = "GDPR_agent"
    BASE_URL_NAME = None # Omit or set to None if calling standard OpenAI API directly

    api_key = dbutils.secrets.get(scope=SECRET_SCOPE, key=API_KEY_NAME)
    
    # Optional: Fetch custom base URL if using a custom gateway/proxy endpoint
    try:
        base_url = dbutils.secrets.get(scope=SECRET_SCOPE, key=BASE_URL_NAME)
    except Exception:
        base_url = None

    # Initialize the client
    openai_client = OpenAI(api_key=api_key, base_url=base_url)
    print("✅ OpenAI client initialized successfully from dbutils secrets.")
except Exception as e:
    print(f"❌ Failed to initialize OpenAI client from secrets: {e}")
    print("⚠️ Fallbacks will trigger automatically during question generation loops.")
    openai_client = None

# =============================================================================
# LLM-Based Question Generation (Adversarial Paraphrasing)
# =============================================================================

def generate_unbiased_question(target_concept: str, source_context: str, question_type: str = "single") -> str:
    """
    Calls your configured OpenAI LLM to generate an adversarial, natural user question
    that completely avoids mentioning the literal string of target_concept.
    """
    if not openai_client:
        return generate_fallback_question(target_concept, source_context, question_type)

    if question_type == "single":
        system_prompt = f"""You are generating evaluation test questions for a GDPR compliance assistant.

CRITICAL CONSTRAINT: Generate a realistic, natural user question about the core themes or requirements covered in '{target_concept}' inside the context of {source_context}. 
You MUST NOT use the literal phrase or words from '{target_concept}' anywhere in your question.

Instead, describe what this concept covers using:
- Synonyms (e.g., "erasing client profiles" instead of "right to erasure")
- Contextual situations (e.g., "what happens when an user demands we remove their data history")
- General compliance language.

The question must sound like an executive or non-technical business user asking for help. 
Generate ONE question sentence only. Do not add intro text, bullet points, or quotes."""

    elif question_type == "multi":
        system_prompt = f"""You are generating an evaluation test question for a GDPR compliance assistant.

CRITICAL CONSTRAINT: Generate a natural user scenario that requires cross-referencing information from multiple sources about '{target_concept}'. 
You MUST NOT use the literal text or words from '{target_concept}' anywhere in your question.

Formulate a multi-part user inquiry that naturally bridges compliance frameworks and empirical tracking. 
Example pattern: "What are the rules regarding notification periods for breaches, and what kind of enforcement penalties have companies run into for dragging their feet?"

Generate ONE question sentence only. Do not add quotes or markdown syntax."""
    
    else:
        system_prompt = f"""Generate a natural user question about '{target_concept}' in {source_context} context, avoiding literal mention of '{target_concept}'."""

    try:
        # Update the model parameter if you are pointing to a specific standard model (e.g., "gpt-4o")
        # or keep your proxy model deployment name if using a Databricks Serving Endpoint route.
        response = openai_client.chat.completions.create(
            model="gpt-4o", 
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": "Generate the question now."}
            ],
            temperature=0.2,
            max_tokens=150
        )
        
        if response.choices and len(response.choices) > 0:
            question = response.choices[0].message.content.strip().strip('"').strip("'")
            
            # Validation safeguard
            if target_concept.lower() in question.lower():
                print(f"⚠️  LLM included literal '{target_concept}' - fallback triggered")
                return generate_fallback_question(target_concept, source_context, question_type)
            return question
        return generate_fallback_question(target_concept, source_context, question_type)
            
    except Exception as e:
        print(f"⚠️  OpenAI LLM call failed ({e}) - using fallback")
        return generate_fallback_question(target_concept, source_context, question_type)


def generate_fallback_question(target_concept: str, source_context: str, question_type: str) -> str:
    if question_type == "single":
        if "Article" in target_concept or "Context" in target_concept:
            return f"What are our baseline transparent communication obligations when handling requests for access to personal information details under {source_context}?"
        elif source_context == "fines":
            return "What major financial penalties have European regulators issued for compliance tracking failures?"
        return "What is our company baseline position on retention timelines for standard behavioral tracking data?"
    return "What are the strict operational notification limits for security incidents, and what fines have been levied for late reports?"


# =============================================================================
# Step 1: Identify and Read Source Tables
# =============================================================================

print("🔍 GENERATING UNBIASED GOLDEN SET FROM SOURCE DATA TABLES")
print("="*80)

LEGISLATION_SOURCE_TABLE = "main.default.gdpr_statutory_legislation" 
FINES_SOURCE_TABLE = "main.default.gdpr_translated_chunks"     
POLICY_SOURCE_TABLE = "main.default.retail_corporate_policy"         

# =============================================================================
# Steps 2, 3, & 4: Data Profiling
# =============================================================================

# Legislation Data Preparation
legislation_articles = []
try:
    legislation_df = spark.table(LEGISLATION_SOURCE_TABLE).limit(500)
    _ = legislation_df.count()
    legislation_df = legislation_df.toPandas()
    if 'article_title' in legislation_df.columns:
        legislation_articles = [a for a in legislation_df['article_title'].dropna().unique().tolist() if a]
except Exception as e:
    print(f"❌ Legislation read error: {e}")

# Fines Data Preparation
companies_found = []
try:
    fines_df = spark.table(FINES_SOURCE_TABLE).limit(500)
    _ = fines_df.count()
    fines_df = fines_df.toPandas()
    companies_set = set()
    if 'translated_text' in fines_df.columns:
        for text in fines_df['translated_text'].dropna():
            for c in ['british airways', 'google', 'amazon', 'meta', 'facebook', 'marriott', 'whatsapp', 'uber']:
                if c in str(text).lower(): companies_set.add(c.title())
    companies_found = sorted(list(companies_set))
except Exception as e:
    print(f"❌ Fines read error: {e}")

# Policy Data Preparation
policy_sections = []
try:
    policy_df = spark.table(POLICY_SOURCE_TABLE).limit(500)
    _ = policy_df.count()
    policy_df = policy_df.toPandas()
    if 'section_title' in policy_df.columns:
        policy_sections = [s for s in policy_df['section_title'].dropna().unique().tolist() if s]
except Exception as e:
    print(f"❌ Policy read error: {e}")

# Fallback populations to guarantee execution arrays are not empty
if not legislation_articles: legislation_articles = ["General Modalities Context", "Right of access"]
if not companies_found: companies_found = ["Facebook", "Google"]
if not policy_sections: policy_sections = ["Global Policy Overview", "Customer Identity Profile"]

# =============================================================================
# Step 5: Loop Architecture to Generate Exactly 30 Cases
# =============================================================================

print("\n" + "="*80)
print("🎯 PROGRAMMATICALLY GENERATING 30 PARAPHRASED TEST CASES")
print("="*80)

test_cases = []

# Target distribution mix for 30 queries
mix_config = [
    {"count": 8, "cat": "single_source_legislation", "ctx": "GDPR legislation", "src": ["legislation"], "type": "single"},
    {"count": 8, "cat": "single_source_fines", "ctx": "GDPR enforcement cases", "src": ["fines"], "type": "single"},
    {"count": 8, "cat": "single_source_policy", "ctx": "internal corporate policy", "src": ["policy"], "type": "single"},
    {"count": 6, "cat": "multi_source_fines_legislation", "ctx": "legislation and fines", "src": ["fines", "legislation"], "type": "multi"}
]

global_id = 1

for config in mix_config:
    for idx in range(config["count"]):
        case_id = f"case_{global_id:03d}"
        
        if config["cat"] == "single_source_legislation":
            concept = legislation_articles[idx % len(legislation_articles)]
            expected = {"sources": config["src"], "must_retrieve_from_articles": [concept], "must_include_in_answer": [concept]}
            eval_method = f"Verify {concept} presence via article_title metadata partition."
            data_src = LEGISLATION_SOURCE_TABLE
            
        elif config["cat"] == "single_source_fines":
            concept = companies_found[idx % len(companies_found)] if idx > 0 else "largest GDPR fines"
            keywords = [concept.lower(), "fine"] if idx > 0 else ["fine", "€", "company"]
            expected = {"sources": config["src"], "must_retrieve_keywords": keywords, "must_include_in_answer": [concept, "fine"]}
            eval_method = "Keyword validation in matched chunks via hybrid tracker text."
            data_src = FINES_SOURCE_TABLE
            
        elif config["cat"] == "single_source_policy":
            concept = policy_sections[idx % len(policy_sections)]
            expected = {"sources": config["src"], "must_retrieve_from_sections": [concept], "must_include_in_answer": ["policy"]}
            eval_method = f"Check if corporate section title '{concept}' matches retrieved records."
            data_src = POLICY_SOURCE_TABLE
            
        elif config["cat"] == "multi_source_fines_legislation":
            leg_concept = legislation_articles[idx % len(legislation_articles)]
            expected = {
                "sources": config["src"],
                "legislation_check": {"must_retrieve_from_articles": [leg_concept]},
                "fines_check": {"must_retrieve_keywords": ["fine", "breach"]},
                "must_include_in_answer": [leg_concept, "fine"]
            }
            concept = f"{leg_concept} compliance alignment"
            eval_method = "Hybrid Evaluation Block: Confirm legislative metadata context and fine records."
            data_src = f"{LEGISLATION_SOURCE_TABLE} & {FINES_SOURCE_TABLE}"

        print(f"🤖 Processing [{case_id}] | Category: {config['cat']}...")
        question_text = generate_unbiased_question(target_concept=concept, source_context=config["ctx"], question_type=config["type"])
        print(f"   ↳ Q: {question_text[:75]}...")

        test_cases.append({
            "id": case_id,
            "question": question_text,
            "category": config["cat"],
            "difficulty": "medium" if global_id % 2 == 0 else "easy",
            "expected_behavior": expected,
            "evaluation_method": eval_method,
            "data_source": f"Confirmed via {data_src}",
            "generation_method": "llm_paraphrased",
            "notes": f"Automated adversarial query targeting conceptual boundary of: {concept}"
        })
        
        global_id += 1

# =============================================================================
# Steps 6 & 7: Structuring and Exporting
# =============================================================================

evaluation_dataset = {
    "metadata": {
        "version": "3.0",
        "created_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "description": "Expanded 30-case evaluation set using OpenAI Client integration via dbutils secrets",
        "generation_method": "source_table_sampling_with_llm_paraphrasing",
        "llm_model": "openai-configured-model",
        "total_cases": len(test_cases),
        "source_tables": {
            "legislation": LEGISLATION_SOURCE_TABLE,
            "fines": FINES_SOURCE_TABLE,
            "policy": POLICY_SOURCE_TABLE
        }
    },
    "test_cases": test_cases
}

base_dir = "/Workspace/Repos/vernonc.lam@gmail.com/GDPR-agent/evaluation_data"
os.makedirs(base_dir, exist_ok=True)
dataset_path = f"{base_dir}/golden_set_v3_production_30.json"

with open(dataset_path, 'w') as f:
    json.dump(evaluation_dataset, f, indent=2)

print("\n" + "="*80)
print(f"🚀 SUCCESS: Production evaluation set locked at {dataset_path}")
print(f"📊 Validated Case Count: {len(test_cases)} / 30 records structured completely.")
print("="*80)